# Import

In [ ]:
import os
import random

import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm 

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch import nn, optim

from sklearn.metrics import log_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Hyperparameter Setting

In [ ]:
# CFG = {
#     'IMG_SIZE': 224,
#     'BATCH_SIZE': 64,
#     'EPOCHS': 10,
#     'LEARNING_RATE': 1e-4,
#     'SEED' : 42
# }

CFG = {
    'IMG_SIZE': 256,                 # ResNet50에 적절한 크기 (성능/속도 균형)
    'BATCH_SIZE': 32,                # 적절한 배치 사이즈 (GPU T4 기준)
    'EPOCHS': 50,                    # 충분한 반복으로 수렴 유도
    'LEARNING_RATE': 1e-3,          # 초기 학습률
    'SEED': 42,                      # 재현성 확보
    'EARLY_STOPPING_PATIENCE': 7,   # 과적합 방지 조기종료
    'MODEL_NAME': 'resnet50',       # ✅ ResNet50으로 변경
    'NUM_CLASSES': 396,             # 클래스 수
    'T_0': 5,                        # CosineAnnealingWarmRestarts 초기 주기
    'T_MULT': 2,                     # Cosine 주기 배수 증가
    'WEIGHT_DECAY': 1e-4,           # 정규화
    
    # 🔥 전략 추가 요소
    'N_FOLDS': 5,                   # KFold 앙상블 수
    'USE_CUTMIX': True,             # CutMix 적용 여부
    'USE_MIXUP': False,             # MixUp은 이번엔 제외
    'USE_TTA': True,                # Test Time Augmentation 사용
    'LABEL_SMOOTHING': 0.1          # Label Smoothing 계수 (정답에 여유를 둠)
}

# Fixed RandomSeed

In [ ]:
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)  # python 내부 해시 고정
    np.random.seed(seed)                      # numpy 고정
    torch.manual_seed(seed)                   # torch 고정
    torch.cuda.manual_seed(seed)              # cuda 고정
    torch.cuda.manual_seed_all(seed)          # multi-GPU 대비
    
    torch.backends.cudnn.deterministic = True  # 완전 결정론적 연산
    torch.backends.cudnn.benchmark = False     # 연산 속도 최적화 off (결정성 위해)
    torch.use_deterministic_algorithms(True)   # 💡 100% 재현성 강제

    # 경고 방지용 (이거 안 쓰면 일부 연산에서 warning 발생)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# ✅ CFG에서 가져와 고정
seed_everything(CFG['SEED'])

# CustomDataset

In [ ]:
import os
from torch.utils.data import Dataset
from PIL import Image

class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, is_test=False):
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.samples = []

        if self.is_test:
            self._load_test_images()
        else:
            self._load_train_images()

    def _load_test_images(self):
        for fname in sorted(os.listdir(self.root_dir)):
            if fname.lower().endswith('.jpg'):
                img_path = os.path.join(self.root_dir, fname)
                self.samples.append((img_path,))

    def _load_train_images(self):
        self.classes = sorted(d for d in os.listdir(self.root_dir) if os.path.isdir(os.path.join(self.root_dir, d)))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        for cls_name in self.classes:
            cls_folder = os.path.join(self.root_dir, cls_name)
            for fname in os.listdir(cls_folder):
                if fname.lower().endswith('.jpg'):
                    img_path = os.path.join(cls_folder, fname)
                    label = self.class_to_idx[cls_name]
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.is_test:
            img_path = self.samples[idx][0]
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image
        else:
            img_path, label = self.samples[idx]
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label

# Data Load

In [ ]:
# !unzip -q /kaggle/input/hecto-ai.zip -d /kaggle/working/
# train_root = '/kaggle/working/hecto-ai/train'

In [ ]:
# import os

# for dirname, _, filenames in os.walk('/kaggle/input'):
#     print(dirname)

In [ ]:
train_root = '/kaggle/input/train'
test_root = '/kaggle/input/test'

In [ ]:
# train_transform = transforms.Compose([
#     transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

# val_transform = transforms.Compose([
#     transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'] + 32, CFG['IMG_SIZE'] + 32)),
    transforms.RandomResizedCrop(CFG['IMG_SIZE'], scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.01),
    transforms.RandomRotation(degrees=5),  # ✅ 보수적으로 줄임
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split

# CutMix 박스 생성 함수
def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

# CutMix Collate 함수
def cutmix_collate_fn(batch, alpha=1.0):
    images, labels = zip(*batch)
    images = torch.stack(images)
    labels = torch.tensor(labels)

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0))
    shuffled_images = images[rand_index]
    shuffled_labels = labels[rand_index]

    bbx1, bby1, bbx2, bby2 = rand_bbox(images.size(), lam)
    images[:, :, bby1:bby2, bbx1:bbx2] = shuffled_images[:, :, bby1:bby2, bbx1:bbx2]

    # 라벨 혼합: (정답 라벨, 섞인 라벨, lambda)
    new_labels = (labels, shuffled_labels, lam)
    return images, new_labels

# 커스텀 이미지 데이터셋 클래스
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, is_test=False):
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.samples = []

        if is_test:
            for fname in sorted(os.listdir(root_dir)):
                if fname.lower().endswith(('.jpg')):
                    img_path = os.path.join(root_dir, fname)
                    self.samples.append((img_path,))
        else:
            self.classes = sorted(os.listdir(root_dir))
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

            for cls_name in self.classes:
                cls_folder = os.path.join(root_dir, cls_name)
                for fname in os.listdir(cls_folder):
                    if fname.lower().endswith(('.jpg')):
                        img_path = os.path.join(cls_folder, fname)
                        label = self.class_to_idx[cls_name]
                        self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.is_test:
            img_path = self.samples[idx][0]
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image
        else:
            img_path, label = self.samples[idx]
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label

# 경로 설정
train_root = '/kaggle/input/hecto-ai/train'

# 전체 데이터셋 로드
full_dataset = CustomImageDataset(train_root, transform=None)
targets = [label for _, label in full_dataset.samples]
class_names = full_dataset.classes  # class index → name 매핑용

# Stratified Train/Val Split
from sklearn.model_selection import train_test_split
train_idx, val_idx = train_test_split(
    range(len(targets)),
    test_size=0.2,
    stratify=targets,
    random_state=CFG['SEED']
)

# Transform (이건 외부에서 정의해줘야 해)
# 👉 예: train_transform, val_transform

# Subset 정의
train_dataset = Subset(CustomImageDataset(train_root, transform=train_transform), train_idx)
val_dataset   = Subset(CustomImageDataset(train_root, transform=val_transform), val_idx)

# DataLoader 정의
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=cutmix_collate_fn  # 🔥 CutMix 적용
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# Model Define

In [ ]:
# class BaseModel(nn.Module):
#     def __init__(self, num_classes):
#         super(BaseModel, self).__init__()
#         self.backbone = models.resnet18(pretrained=False)  # ResNet18 모델 불러오기
#         self.feature_dim = self.backbone.fc.in_features 
#         self.backbone.fc = nn.Identity()  # feature extractor로만 사용
#         self.head = nn.Linear(self.feature_dim, num_classes)  # 분류기

#     def forward(self, x):
#         x = self.backbone(x)       
#         x = self.head(x) 
#         return x

import torch.nn as nn
import torchvision.models as models

class BaseModel(nn.Module):
    def __init__(self, num_classes):
        super(BaseModel, self).__init__()
        # ✅ 사전학습된 ResNet50 사용
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        # 기존 classifier 제거하고 feature extractor로만 사용
        self.feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        # ✅ Regularization
        self.dropout = nn.Dropout(p=0.5)
        self.bn = nn.BatchNorm1d(self.feature_dim)

        # ✅ 분류 헤드
        self.head = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.dropout(x)
        x = self.bn(x)
        x = self.head(x)
        return x

# Train/ Validation

In [ ]:
# model = BaseModel(num_classes=len(class_names)).to(device)
# best_logloss = float('inf')

# # 손실 함수
# criterion = nn.CrossEntropyLoss()

# # 옵티마이저
# optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])

# # 학습 및 검증 루프
# for epoch in range(CFG['EPOCHS']):
#     # Train
#     model.train()
#     train_loss = 0.0
#     for images, labels in tqdm(train_loader, desc=f"[Epoch {epoch+1}/{CFG['EPOCHS']}] Training"):
#         images, labels = images.to(device), labels.to(device)
#         optimizer.zero_grad()
#         outputs = model(images)  # logits
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         train_loss += loss.item()

#     avg_train_loss = train_loss / len(train_loader)

#     # Validation
#     model.eval()
#     val_loss = 0.0
#     correct = 0
#     total = 0
#     all_probs = []
#     all_labels = []

#     with torch.no_grad():
#         for images, labels in tqdm(val_loader, desc=f"[Epoch {epoch+1}/{CFG['EPOCHS']}] Validation"):
#             images, labels = images.to(device), labels.to(device)
#             outputs = model(images)
#             loss = criterion(outputs, labels)
#             val_loss += loss.item()

#             # Accuracy
#             _, preds = torch.max(outputs, 1)
#             correct += (preds == labels).sum().item()
#             total += labels.size(0)

#             # LogLoss
#             probs = F.softmax(outputs, dim=1)
#             all_probs.extend(probs.cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())

#     avg_val_loss = val_loss / len(val_loader)
#     val_accuracy = 100 * correct / total
#     val_logloss = log_loss(all_labels, all_probs, labels=list(range(len(class_names))))

#     # 결과 출력
#     print(f"Train Loss : {avg_train_loss:.4f} || Valid Loss : {avg_val_loss:.4f} | Valid Accuracy : {val_accuracy:.4f}%")

#     # Best model 저장
#     if val_logloss < best_logloss:
#         best_logloss = val_logloss
#         torch.save(model.state_dict(), f'best_model.pth')
#         print(f"📦 Best model saved at epoch {epoch+1} (logloss: {val_logloss:.4f})")

from sklearn.metrics import log_loss
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report
import os
import torch.nn.functional as F

model = BaseModel(num_classes=len(class_names)).to(device)
best_logloss = float('inf')
start_epoch = 0

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'])
scaler = GradScaler()

checkpoint_path = 'checkpoint_latest.pth'
if os.path.exists(checkpoint_path):
    print("🔄 이전 체크포인트 불러오는 중...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    scaler.load_state_dict(checkpoint['scaler'])
    start_epoch = checkpoint['epoch'] + 1
    best_logloss = checkpoint['best_logloss']
    print(f"✅ {start_epoch} 에폭부터 재개합니다.")

for epoch in range(start_epoch, CFG['EPOCHS']):
    print(f"\n📘 Epoch {epoch+1}/{CFG['EPOCHS']}")

    # ---------- Train ----------
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training"):
        images = images.to(device)

        # ✅ CutMix 여부에 따라 라벨 분기
        if isinstance(labels, tuple):
            targets1, targets2, lam = labels
            targets1 = targets1.to(device)
            targets2 = targets2.to(device)
        else:
            labels = labels.to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            if isinstance(labels, tuple):
                loss = lam * criterion(outputs, targets1) + (1 - lam) * criterion(outputs, targets2)
            else:
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ---------- Validation ----------
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"[Epoch {epoch+1}] Validation"):
            images = images.to(device)
            labels = labels.to(device)

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            probs = F.softmax(outputs, dim=1)
            probs = torch.clamp(probs, 1e-7, 1 - 1e-7)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total
    val_logloss = log_loss(all_labels, all_probs, labels=list(range(len(class_names))))

    print(f"✅ Train Loss : {avg_train_loss:.4f} | Valid Loss : {avg_val_loss:.4f} | Valid Acc : {val_accuracy:.2f}% | LogLoss : {val_logloss:.4f}")

    # ---------- Save Best ----------
    if val_logloss < best_logloss:
        best_logloss = val_logloss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"💾 Best model saved at epoch {epoch+1} (logloss: {val_logloss:.4f})")

    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_logloss': best_logloss
    }, checkpoint_path)

    scheduler.step()


# Inference

In [ ]:
test_dataset = CustomImageDataset(test_root, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from PIL import Image
import numpy as np

# 테스트셋 로드
test_dataset = CustomImageDataset(test_root, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

# 모델 로드
model = BaseModel(num_classes=len(class_names))
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.to(device)
model.eval()

# ✅ TTA 횟수 설정
N_TTA = 5
results = []

with torch.no_grad():
    for images in tqdm(test_loader, desc="TTA Inference"):
        images = images.to(device)
        batch_size = images.size(0)

        # 확률 누적용 텐서
        tta_probs = torch.zeros(batch_size, len(class_names)).to(device)

        for _ in range(N_TTA):
            # ✅ TTA를 위한 augmentation 적용
            augmented = images  # val_transform이 DataLoader에서 이미 적용되었으므로 생략 가능
            outputs = model(augmented)
            probs = F.softmax(outputs, dim=1)
            probs = torch.clamp(probs, 1e-7, 1 - 1e-7)  # 안정성
            tta_probs += probs

        avg_probs = tta_probs / N_TTA

        for prob in avg_probs.cpu():
            result = {
                class_names[i]: prob[i].item()
                for i in range(len(class_names))
            }
            results.append(result)

# ✅ DataFrame 변환 및 저장
pred = pd.DataFrame(results)
pred.index.name = 'id'  # 제출 형식에 맞춰 인덱스가 id 역할이면
# pred.to_csv('submission.csv')  # 필요 시 저장


# Submission

In [ ]:
# pred 컬럼 순서를 sample_submission 기준으로 재정렬
submission = pd.read_csv('/kaggle/input/sample_submission.csv', encoding='utf-8-sig')

class_columns = [col for col in submission.columns if col != 'ID']  # 'ID' 제외
pred = pred[class_columns]  # 예측 결과도 그 순서로 정렬

submission[class_columns] = pred.values
submission.to_csv('submission.csv', index=False, encoding='utf-8-sig')